# TechArena CTR: full-population grouped ranker
Both private prepared datasets required. All three prior dates,600 fixed GPU pairwise-ranking rounds, ten candidates per query and history-population query weights.
Candidate softmax is multiplied by the existing eight-epoch neural003 event-click mass. Fixed10% rank blend with V53. Both-date gains versus V53 and neural003 required, with exact checkpoint replay.
Earlier ranker used30k sampled clicked queries. This trial uses the full chronological corpus. Reused development cohorts; no official forecast or CSV export.


In [ ]:
from pathlib import Path
import hashlib, importlib.util
import torch
assert torch.cuda.is_available(), "Enable Kaggle GPU"
roots=list(Path("/kaggle/input").rglob("bundle_manifest.json"))
neural_roots=list(Path("/kaggle/input").rglob("neural_manifest.json"))
assert len(roots)==len(neural_roots)==1
DATA_ROOT=roots[0].parent
NEURAL_ROOT=neural_roots[0].parent
weighting_path=Path("/kaggle/working/tabular_choice_weighting.py")
weighting_path.write_text("\"\"\"Fixed-capacity CatBoost GPU expert on the existing causal feature bundle.\r\n\r\nNo evaluation labels enter fitting or early stopping. Main day6 is followed by\r\nunchanged day5 confirmation only if its conditional gain gate passes.\r\n\"\"\"\r\nimport gc\r\nimport importlib.util\r\nimport json\r\nimport time\r\nfrom pathlib import Path\r\n\r\nimport catboost\r\nimport numpy as np\r\nfrom catboost import CatBoostClassifier, Pool\r\n\r\nPARAMS=dict(iterations=1600,depth=8,learning_rate=.04,l2_leaf_reg=10.,\r\n            loss_function=\"Logloss\",task_type=\"GPU\",devices=\"0\",border_count=63,\r\n            bootstrap_type=\"Bernoulli\",subsample=.8,random_seed=6801,\r\n            boosting_type=\"Plain\",thread_count=4,allow_writing_files=False,\r\n            verbose=100)\r\nBLEND=.25\r\n\r\n\r\ndef helpers(root):\r\n    spec=importlib.util.spec_from_file_location(\"ctr_cb_gpu_helpers\",Path(root)/\"runner.py\")\r\n    module=importlib.util.module_from_spec(spec)\r\n    spec.loader.exec_module(module)\r\n    return module\r\n\r\n\r\ndef predict(model,features):\r\n    out=[]\r\n    for off in range(0,len(features),25000):\r\n        x=features.block(off,min(off+25000,len(features)))\r\n        out.append(model.predict_proba(x,thread_count=4)[:,1])\r\n    return np.concatenate(out)\r\n\r\n\r\ndef training_weights(neural_root, day, labels, h):\r\n    \"\"\"Label-free target history distribution, with exact fitting-row alignment.\"\"\"\r\n    neural_root=Path(neural_root)\r\n    nm=json.loads((neural_root/\"neural_manifest.json\").read_text())\r\n    for name in [\"events.npz\", \"val5.npz\", \"val6.npz\"]:\r\n        assert h.digest(neural_root/name)==nm[\"files\"][name][\"sha256\"]\r\n    with np.load(neural_root/\"events.npz\") as z:\r\n        timestamp=z[\"timestamp\"].copy()\r\n        target=z[\"target\"].copy()\r\n        bins=np.digitize((z[\"hist\"]>=0).sum(1),[0,4,9,19],right=True)\r\n    shares=[]\r\n    for d in [5,6]:\r\n        with np.load(neural_root/f\"val{d}.npz\") as z:\r\n            b=np.repeat(bins[z[\"indices\"]],10)\r\n            w=z[\"weights\"]\r\n            share=np.bincount(b,weights=w,minlength=5)/w.sum()\r\n            shares.append(share)\r\n    assert np.allclose(shares[0],shares[1],atol=1e-7)\r\n    target_share=shares[0]\r\n    event_day=(timestamp-int(nm[\"start\"]))//86400\r\n    selected=(event_day>=day-3)&(event_day<day)\r\n    expected=(target[selected,None]==np.arange(10)[None,:]).astype(\"float32\").ravel()\r\n    assert np.array_equal(expected,labels), \"Fitting labels and event chronology differ\"\r\n    assert (timestamp[selected]<int(nm[\"start\"])+day*86400).all()\r\n    b=np.repeat(bins[selected],10)\r\n    source_share=np.bincount(b,minlength=5)/len(b)\r\n    assert (source_share>0).all()\r\n    weights=(target_share/source_share)[b].astype(\"float32\")\r\n    weights/=weights.mean()\r\n    assert np.isfinite(weights).all() and (weights>0).all()\r\n    return weights,dict(target_history_share=target_share.tolist(),\r\n                        training_history_share=source_share.tolist(),\r\n                        weight_min=float(weights.min()),weight_max=float(weights.max()),\r\n                        alignment=\"exact chronological labels; strictly pre-cutoff events\",\r\n                        population=\"unlabeled evaluation history distribution reconstructed from fixed weights\")\r\n\r\n\r\ndef fit(root,output,manifest,day,h,neural_root):\r\n    started=time.monotonic()\r\n    output.mkdir(parents=True,exist_ok=False)\r\n    days=list(range(day-3,day))\r\n    validation=manifest[\"validation\"][str(day)]\r\n    specs=[manifest[\"train\"][str(d)] for d in days]\r\n    labels=[]\r\n    for spec in specs:\r\n        meta=json.loads((root/spec[\"metadata\"]).read_text())\r\n        assert meta[\"last_timestamp\"]<validation[\"cutoff\"]\r\n        labels.append(np.load(root/spec[\"labels\"]))\r\n    y=np.concatenate(labels).astype(\"float32\")\r\n    weights,weight_recipe=training_weights(neural_root,day,y,h)\r\n    feature=h.FeatureBlocks(root,[s[\"features\"] for s in specs])\r\n    assert len(feature)==len(y)\r\n    recipe=dict(history_importance=weight_recipe,parameters=PARAMS,blend_weight=BLEND,day=day,fit_days=days,\r\n                fit_rows=len(y),columns=354,validation_cutoff=validation[\"cutoff\"],\r\n                input_manifest_sha256=h.digest(root/\"bundle_manifest.json\"),\r\n                source_sha256=h.digest(Path(__file__)),catboost=catboost.__version__,\r\n                selection=\"fixed before scoring; no eval_set or early stopping\")\r\n    (output/\"recipe.json\").write_text(json.dumps(recipe,indent=2))\r\n    temporary=output/\"fit_features.npy\"\r\n    assert temporary.resolve().parent==output.resolve()\r\n    matrix=np.lib.format.open_memmap(temporary,mode=\"w+\",dtype=\"float32\",shape=(len(y),354))\r\n    for off in range(0,len(y),50000):\r\n        end=min(off+50000,len(y));matrix[off:end]=feature.block(off,end)\r\n        if off%500000==0:print(dict(stage=\"prepare_features\",day=day,rows=end,total=len(y)),flush=True)\r\n    matrix.flush()\r\n    pool=Pool(matrix,label=y,weight=weights,thread_count=4)\r\n    model=CatBoostClassifier(**PARAMS)\r\n    print(dict(stage=\"catboost_gpu_fit\",day=day,rows=len(y),iterations=PARAMS[\"iterations\"]),flush=True)\r\n    model.fit(pool)\r\n    model_path=output/\"model.cbm\"\r\n    model.save_model(str(model_path))\r\n    del pool\r\n    matrix._mmap.close()\r\n    del matrix,y,weights\r\n    gc.collect()\r\n    # Remove only the scratch array authored by this fresh experiment.\r\n    assert temporary.resolve().parent==output.resolve()\r\n    temporary.unlink()\r\n    with np.load(root/validation[\"data\"]) as z:\r\n        val={k:z[k].copy() for k in z.files}\r\n    xv=h.FeatureBlocks(root,[validation[\"features\"]])\r\n    assert len(xv)==len(val[\"y\"]) and len(xv)%10==0\r\n    raw=predict(model,xv)\r\n    assert np.isfinite(raw).all() and ((raw>=0)&(raw<=1)).all()\r\n    pred=h.rank((1-BLEND)*h.rank(val[\"baseline\"])+BLEND*h.rank(raw))\r\n    m=h.metrics(val[\"y\"],pred,val[\"baseline\"],val[\"weights\"])\r\n    restored=CatBoostClassifier()\r\n    restored.load_model(str(model_path))\r\n    replay=predict(restored,xv)\r\n    replay_error=float(np.max(np.abs(replay-raw)))\r\n    assert replay_error<1e-12,replay_error\r\n    m.update(day=day,raw_auc=float(h.roc_auc_score(val[\"y\"],raw)),rows=len(raw),\r\n             saved_model_replay_max_error=replay_error,saved_checkpoint_replay=\"PASS\",\r\n             model_sha256=h.digest(model_path),elapsed_seconds=round(time.monotonic()-started,1),\r\n             official_auc=None,csv_ready=False)\r\n    np.savez_compressed(output/\"predictions.npz\",raw=raw,pred=pred,**val)\r\n    (output/\"report.json\").write_text(json.dumps(m,indent=2))\r\n    print(json.dumps(m,indent=2),flush=True)\r\n    return m\r\n\r\n\r\ndef run(root,output,neural_root):\r\n    root,output=Path(root),Path(output)\r\n    h=helpers(root)\r\n    manifest=h.load_bundle(root,verify=True)\r\n    output.mkdir(parents=True,exist_ok=False)\r\n    (output/\"catboost_gpu_weighted_runner.py\").write_text(Path(__file__).read_text())\r\n    reports=[fit(root,output/\"day6\",manifest,6,h,neural_root)]\r\n    if reports[0][\"positive_gate\"]:\r\n        reports.append(fit(root,output/\"day5\",manifest,5,h,neural_root))\r\n    eligible=len(reports)==2 and all(m[\"positive_gate\"] for m in reports)\r\n    summary=dict(reports=reports,eligible_for_review=eligible,target=.83,\r\n                 target_achieved=bool(eligible and reports[0][\"history_adjusted_auc\"]>=.83),\r\n                 csv_ready=False,official_auc=None,\r\n                 note=\"Same reused main dates; fixed history-weighted CatBoost GPU expert. Local metric, not official forecast.\")\r\n    (output/\"summary.json\").write_text(json.dumps(summary,indent=2))\r\n    return summary\r\n",encoding="utf-8")
assert hashlib.sha256(weighting_path.read_bytes()).hexdigest()=="133e47d002c174d6f15c090f2fffc7f0ef63e4b287f3cc6b1d9a30b02e7b2d9c"
source_path=Path("/kaggle/working/tabular_choice_runner.py")
source_path.write_text("\"\"\"Permutation-equivariant neural choice expert over causal numeric features.\r\n\r\nFitting and normalization use prior dates only. A fixed blend is scored on\r\nday6 and confirmed unchanged on day5 only after a positive gain gate.\r\n\"\"\"\r\nimport gc\r\nimport importlib.util\r\nimport json\r\nimport time\r\nfrom pathlib import Path\r\n\r\nimport numpy as np\r\nimport torch\r\nfrom torch import nn\r\n\r\nRECIPE = dict(epochs=8, batch_size=1024, learning_rate=.0005,\r\n              weight_decay=.005, dropout=.15, seed=6802, blend_weight=.1,\r\n              hidden=128, representation=64,\r\n              feature_transform='Within-event standardization of varying columns; fitting-standardized shared columns')\r\n\r\n\r\ndef load_module(name, path):\r\n    spec = importlib.util.spec_from_file_location(name, path)\r\n    module = importlib.util.module_from_spec(spec)\r\n    spec.loader.exec_module(module)\r\n    return module\r\n\r\n\r\nclass ChoiceNet(nn.Module):\r\n    def __init__(self, mean, scale, recipe):\r\n        super().__init__()\r\n        self.register_buffer('mean', mean.clone())\r\n        self.register_buffer('scale', scale.clone().clamp_min(.01))\r\n        self.register_buffer('active', (scale > 1e-8).float())\r\n        hidden, dim = recipe['hidden'], recipe['representation']\r\n        self.encoder = nn.Sequential(nn.Linear(len(mean), hidden), nn.ReLU(),\r\n                                     nn.Linear(hidden, dim), nn.ReLU())\r\n        self.candidate = nn.Sequential(nn.Linear(3*dim, 64), nn.ReLU(),\r\n                                       nn.Dropout(recipe['dropout']), nn.Linear(64, 1))\r\n        self.outside = nn.Sequential(nn.Linear(2*dim, 32), nn.ReLU(), nn.Linear(32, 1))\r\n        # Sensible initial no-click mass; no dependency on candidate position.\r\n        nn.init.constant_(self.candidate[-1].bias, -2.)\r\n        nn.init.zeros_(self.outside[-1].bias)\r\n\r\n    def forward(self, x):\r\n        x = ((x-self.mean)/self.scale)*self.active\r\n        event_variance, event_mean = torch.var_mean(x, dim=1, keepdim=True, unbiased=False)\r\n        relative = (x-event_mean)/event_variance.sqrt().clamp_min(1e-4)\r\n        x = torch.where(event_variance > 1e-8, relative.clamp(-5, 5), x.clamp(-10, 10))\r\n        z = self.encoder(x)\r\n        avg, peak = z.mean(1), z.max(1).values\r\n        context = torch.cat([avg, peak], -1)\r\n        combined = torch.cat([z, context[:, None, :].expand(-1, 10, -1)], -1)\r\n        return torch.cat([self.candidate(combined).squeeze(-1), self.outside(context)], -1)\r\n\r\n\r\n@torch.no_grad()\r\ndef predict(model, features, device):\r\n    model.eval()\r\n    result = []\r\n    for off in range(0, len(features), 20000):\r\n        x = torch.as_tensor(features.block(off, min(off+20000, len(features))), device=device)\r\n        p = model(x.reshape(-1, 10, 354)).softmax(1)[:, :10]\r\n        result.append(p.cpu().numpy().ravel())\r\n    return np.concatenate(result)\r\n\r\n\r\ndef fit(root, neural_root, output, manifest, day, h, weighting, device):\r\n    started = time.monotonic()\r\n    output.mkdir(parents=True, exist_ok=False)\r\n    torch.manual_seed(RECIPE['seed'])\r\n    specs = [manifest['train'][str(d)] for d in range(day-3, day)]\r\n    validation = manifest['validation'][str(day)]\r\n    labels = []\r\n    for s in specs:\r\n        meta = json.loads((root/s['metadata']).read_text())\r\n        assert meta['last_timestamp'] < validation['cutoff']\r\n        labels.append(np.load(root/s['labels']))\r\n    y = np.concatenate(labels).astype('float32')\r\n    assert len(y) % 10 == 0\r\n    event_y = y.reshape(-1, 10)\r\n    assert ((event_y == 0) | (event_y == 1)).all() and (event_y.sum(1) <= 1).all()\r\n    target = np.where(event_y.sum(1) > 0, event_y.argmax(1), 10)\r\n    weights, weight_recipe = weighting.training_weights(neural_root, day, y, h)\r\n    weights = weights.reshape(-1, 10)\r\n    assert np.array_equal(weights, np.repeat(weights[:, :1], 10, axis=1))\r\n    features = h.FeatureBlocks(root, [s['features'] for s in specs])\r\n    assert len(features) == len(y)\r\n    # Transfer one block at a time. No second full host feature matrix.\r\n    x = torch.empty((len(y), 354), device=device)\r\n    for off in range(0, len(y), 50000):\r\n        end = min(off+50000, len(y))\r\n        x[off:end] = torch.as_tensor(features.block(off, end), device=device)\r\n    variance, mean = torch.var_mean(x, dim=0, unbiased=False)\r\n    scale = variance.sqrt()\r\n    assert torch.isfinite(mean).all() and torch.isfinite(scale).all()\r\n    x = x.reshape(-1, 10, 354)\r\n    target = torch.as_tensor(target, device=device, dtype=torch.long)\r\n    weight = torch.as_tensor(weights[:, 0].copy(), device=device)\r\n    model = ChoiceNet(mean, scale, RECIPE).to(device)\r\n    optimizer = torch.optim.AdamW(model.parameters(), lr=RECIPE['learning_rate'],\r\n                                  weight_decay=RECIPE['weight_decay'])\r\n    recipe = dict(recipe=RECIPE, day=day, fit_days=list(range(day-3, day)),\r\n                  fit_rows=len(y), validation_cutoff=validation['cutoff'],\r\n                  weighting=weight_recipe, source_sha256=h.digest(Path(__file__)),\r\n                  manifest_sha256=h.digest(root/'bundle_manifest.json'),\r\n                  torch_version=torch.__version__,\r\n                  constant_columns=int((scale <= 1e-8).sum()),\r\n                  constant_feature_policy='Zero at fitting and inference; no learned signal from invariant fitting columns',\r\n                  selection='Fixed epochs and blend before scoring; no validation optimization')\r\n    (output/'recipe.json').write_text(json.dumps(recipe, indent=2))\r\n    for epoch in range(RECIPE['epochs']):\r\n        model.train()\r\n        order = torch.randperm(len(x), device=device)\r\n        total = mass = 0.\r\n        for off in range(0, len(x), RECIPE['batch_size']):\r\n            ix = order[off:off+RECIPE['batch_size']]\r\n            optimizer.zero_grad(set_to_none=True)\r\n            losses = nn.functional.cross_entropy(model(x[ix]), target[ix], reduction='none')\r\n            loss = (losses*weight[ix]).sum()/weight[ix].sum()\r\n            loss.backward()\r\n            nn.utils.clip_grad_norm_(model.parameters(), 5.)\r\n            optimizer.step()\r\n            total += float((losses.detach()*weight[ix]).sum())\r\n            mass += float(weight[ix].sum())\r\n        print(dict(day=day, epoch=epoch+1, epochs=RECIPE['epochs'],\r\n                   loss=total/mass, elapsed_seconds=round(time.monotonic()-started, 1)), flush=True)\r\n    checkpoint = output/'model.pt'\r\n    torch.save(model.state_dict(), checkpoint)\r\n    del x, target, weight, optimizer\r\n    gc.collect()\r\n    if device == 'cuda': torch.cuda.empty_cache()\r\n    with np.load(root/validation['data']) as z:\r\n        val = {key:z[key].copy() for key in z.files}\r\n    xv = h.FeatureBlocks(root, [validation['features']])\r\n    assert len(xv) == len(val['y'])\r\n    raw = predict(model, xv, device)\r\n    assert np.isfinite(raw).all() and ((raw >= 0) & (raw <= 1)).all()\r\n    assert (raw.reshape(-1, 10).sum(1) <= 1.00001).all()\r\n    pred = h.rank((1-RECIPE['blend_weight'])*h.rank(val['baseline'])\r\n                  + RECIPE['blend_weight']*h.rank(raw))\r\n    m = h.metrics(val['y'], pred, val['baseline'], val['weights'])\r\n    restored = ChoiceNet(mean, scale, RECIPE).to(device)\r\n    restored.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=True))\r\n    replay = predict(restored, xv, device)\r\n    error = float(np.max(np.abs(raw-replay)))\r\n    assert error < 1e-7, error\r\n    m.update(day=day, raw_auc=float(h.roc_auc_score(val['y'], raw)),\r\n             saved_model_replay_max_error=error, saved_checkpoint_replay='PASS',\r\n             model_sha256=h.digest(checkpoint), rows=len(raw),\r\n             elapsed_seconds=round(time.monotonic()-started, 1), csv_ready=False, official_auc=None)\r\n    np.savez_compressed(output/'predictions.npz', raw=raw, pred=pred, **val)\r\n    (output/'report.json').write_text(json.dumps(m, indent=2))\r\n    print(json.dumps(m, indent=2), flush=True)\r\n    del model, restored\r\n    gc.collect()\r\n    if device == 'cuda': torch.cuda.empty_cache()\r\n    return m\r\n\r\n\r\ndef run(root, neural_root, output, weighting_path, device='cuda'):\r\n    root, neural_root, output = Path(root), Path(neural_root), Path(output)\r\n    assert device == 'cpu' or torch.cuda.is_available()\r\n    torch.set_num_threads(4)\r\n    h = load_module('tabular_choice_helpers', root/'runner.py')\r\n    weighting = load_module('tabular_choice_weights', weighting_path)\r\n    manifest = h.load_bundle(root, verify=True)\r\n    output.mkdir(parents=True, exist_ok=False)\r\n    (output/'tabular_choice_setwise_runner.py').write_text(Path(__file__).read_text())\r\n    (output/'weighting_source.py').write_text(Path(weighting_path).read_text())\r\n    reports = [fit(root, neural_root, output/'day6', manifest, 6, h, weighting, device)]\r\n    if reports[0]['positive_gate']:\r\n        reports.append(fit(root, neural_root, output/'day5', manifest, 5, h, weighting, device))\r\n    eligible = len(reports) == 2 and all(r['positive_gate'] for r in reports)\r\n    summary = dict(reports=reports, eligible_for_review=eligible, target=.83,\r\n                   target_achieved=bool(eligible and reports[0]['history_adjusted_auc'] >= .83),\r\n                   csv_ready=False, official_auc=None,\r\n                   note='Reused main validation dates. A local metric, not an official forecast.')\r\n    (output/'summary.json').write_text(json.dumps(summary, indent=2))\r\n    return summary\r\n",encoding="utf-8")
assert hashlib.sha256(source_path.read_bytes()).hexdigest()=="520e2dae7b3257cb5db27fbada7fda9fb069b2f08add73024eeee1be96babaf6"
rank_path=Path("/kaggle/working/full_rank_runner.py")
rank_path.write_text("\"\"\"Full-population grouped ranking with a causal neural event-click mass.\n\nDifferent from the earlier 30k-event sampled ranker: all prior-date queries,\nhistory-population query weights and a fixed GPU recipe. No held-out tuning.\n\"\"\"\nimport gc\nimport importlib.util\nimport json\nimport shutil\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport xgboost as xgb\n\nROUNDS = 600\nPARAMS = dict(objective='rank:pairwise', tree_method='hist', max_depth=6,\n              max_bin=63, eta=.05, min_child_weight=50, subsample=.8,\n              colsample_bytree=.9, reg_lambda=10., reg_alpha=.1,\n              nthread=4, seed=6901, max_cached_hist_node=32)\n\n\ndef module(name, path):\n    spec = importlib.util.spec_from_file_location(name, path)\n    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)\n    return m\n\n\ndef probabilities(scores, mass):\n    scores = np.asarray(scores).reshape(-1,10).astype('float64')\n    mass = np.asarray(mass).reshape(-1)\n    assert len(scores) == len(mass) and np.isfinite(scores).all()\n    assert np.isfinite(mass).all() and ((mass >= 0) & (mass <= 1)).all()\n    exp = np.exp(scores-scores.max(1,keepdims=True))\n    return (exp/exp.sum(1,keepdims=True)*mass[:,None]).ravel()\n\n\ndef fit(root, neural_root, output, manifest, day, h, neural, weighting, neural_output, device):\n    output.mkdir(parents=True, exist_ok=False)\n    specs = [manifest['train'][str(d)] for d in range(day-3,day)]\n    validation = manifest['validation'][str(day)]\n    labels = []\n    for spec in specs:\n        meta = json.loads((root/spec['metadata']).read_text())\n        assert meta['last_timestamp'] < validation['cutoff']\n        labels.append(np.load(root/spec['labels']))\n    y = np.concatenate(labels).astype('float32')\n    assert len(y)%10 == 0 and ((y == 0) | (y == 1)).all()\n    assert (y.reshape(-1,10).sum(1) <= 1).all()\n    weights, weight_recipe = weighting.training_weights(neural_root,day,y,h)\n    assert np.array_equal(weights.reshape(-1,10),np.repeat(weights.reshape(-1,10)[:,:1],10,axis=1))\n    features = h.FeatureBlocks(root,[s['features'] for s in specs])\n    assert len(features) == len(y)\n    dev, cp = h.choose_device(device)\n    matrix = xgb.QuantileDMatrix(h.BatchIterator(features,y,cp),nthread=4,max_bin=63)\n    matrix.set_group(np.full(len(y)//10,10,dtype='uint32'))\n    matrix.set_weight(weights[::10])\n    recipe = dict(params=dict(PARAMS,device=dev),rounds=ROUNDS,fit_rows=len(y),\n                  fit_queries=len(y)//10,fit_days=list(range(day-3,day)),\n                  clicked_queries=int((y.reshape(-1,10).sum(1)>0).sum()),\n                  validation_cutoff=validation['cutoff'],weighting=weight_recipe,\n                  blend_weight=.1,source_sha256=h.digest(Path(__file__)),\n                  manifest_sha256=h.digest(root/'bundle_manifest.json'),\n                  xgboost=xgb.__version__,\n                  selection='Fixed full-population ranking; no early stopping or validation-label training')\n    (output/'recipe.json').write_text(json.dumps(recipe,indent=2))\n    print(dict(stage='fit_full_ranker',day=day,rows=len(y),queries=len(y)//10),flush=True)\n    model = xgb.train(recipe['params'],matrix,num_boost_round=ROUNDS,\n                      callbacks=[h.Progress(output,ROUNDS)])\n    checkpoint = output/'model.ubj'; model.save_model(checkpoint)\n    del matrix,y,features\n    gc.collect()\n    if cp is not None: cp.get_default_memory_pool().free_all_blocks()\n    val = dict(np.load(root/validation['data']))\n    xv = h.FeatureBlocks(root,[validation['features']])\n    def predict(m):\n        result=[]\n        for off in range(0,len(xv),25000):\n            x=xv.block(off,min(off+25000,len(xv)))\n            p=m.inplace_predict(cp.asarray(x) if cp is not None else x)\n            result.append(cp.asnumpy(p) if cp is not None else p)\n        return np.concatenate(result)\n    scores = predict(model)\n    restored = xgb.Booster(); restored.load_model(checkpoint)\n    restored.set_param({'device':dev,'nthread':4})\n    assert np.array_equal(predict(restored),scores)\n    directory = neural_output/f'day{day}'\n    nr = json.loads((directory/'report.json').read_text())\n    assert h.digest(directory/'model.pt') == nr['model_sha256']\n    nn = dict(np.load(directory/'predictions.npz'))\n    for key in ['y','events','baseline','weights']:\n        assert np.array_equal(nn[key],val[key]),key\n    state = torch.load(directory/'model.pt',map_location=dev,weights_only=True)\n    nm = neural.ChoiceNet(state['mean'],state['scale'],neural.RECIPE).to(dev)\n    nm.load_state_dict(state)\n    assert np.array_equal(neural.predict(nm,xv,dev),nn['raw'])\n    mass = np.clip(nn['raw'].reshape(-1,10).sum(1),0,1)\n    raw = probabilities(scores,mass)\n    pred = h.rank(.9*h.rank(val['baseline'])+.1*h.rank(raw))\n    m = h.metrics(val['y'],pred,val['baseline'],val['weights'])\n    versus_single = h.metrics(val['y'],pred,nn['pred'],val['weights'])\n    m.update(day=day,versus_single=versus_single,raw_auc=float(h.roc_auc_score(val['y'],raw)),\n             saved_checkpoint_replay='PASS, ranker and neural exact',rows=len(raw),\n             model_sha256=h.digest(checkpoint),csv_ready=False,official_auc=None)\n    np.savez_compressed(output/'predictions.npz',raw=raw,pred=pred,scores=scores,event_mass=mass,**val)\n    shutil.copy2(directory/'model.pt',output/'neural_mass_model.pt')\n    shutil.copy2(directory/'recipe.json',output/'neural_mass_recipe.json')\n    (output/'report.json').write_text(json.dumps(m,indent=2))\n    print('FULL_RANK_RESULT',json.dumps(m,indent=2),flush=True)\n    return m\n\n\ndef run(root,neural_root,output,model_source,weighting_source,neural_output,device='cuda'):\n    root,neural_root,output,neural_output=map(Path,[root,neural_root,output,neural_output])\n    h=module('full_rank_helpers',root/'runner.py')\n    neural=module('full_rank_neural',model_source)\n    weighting=module('full_rank_weights',weighting_source)\n    assert neural.RECIPE['epochs']==8 and neural.RECIPE['seed']==6802\n    manifest=h.load_bundle(root,verify=True)\n    if not (neural_output/'summary.json').exists():\n        neural.run(root,neural_root,neural_output,weighting_source,device)\n    assert json.loads((neural_output/'summary.json').read_text())['eligible_for_review']\n    output.mkdir(parents=True,exist_ok=False)\n    for src,name in [(Path(__file__),'full_rank_runner.py'),(Path(model_source),'neural_source.py'),\n                     (Path(weighting_source),'weighting_source.py')]:\n        shutil.copy2(src,output/name)\n    reports=[]\n    for day in [6,5]:\n        r=fit(root,neural_root,output/f'day{day}',manifest,day,h,neural,weighting,neural_output,device)\n        reports.append(r)\n        if not (r['positive_gate'] and r['versus_single']['positive_gate']):break\n    eligible=len(reports)==2 and all(r['positive_gate'] and r['versus_single']['positive_gate'] for r in reports)\n    summary=dict(reports=reports,eligible_for_review=eligible,target=.83,\n                 target_achieved=bool(eligible and reports[0]['history_adjusted_auc']>=.83),\n                 csv_ready=False,official_auc=None,note='Reused development dates; no official forecast.')\n    (output/'summary.json').write_text(json.dumps(summary,indent=2))\n    return summary\n",encoding="utf-8")
assert hashlib.sha256(rank_path.read_bytes()).hexdigest()=="df8b2dd2a198d27023836b147d43f3940b2016a658506d7d648d2d84b025fe8d"
spec=importlib.util.spec_from_file_location("ctr_full_rank",rank_path)
full_rank=importlib.util.module_from_spec(spec)
spec.loader.exec_module(full_rank)
RUN_NAME="ctr-full-rank-001"
output=Path("/kaggle/working")/RUN_NAME
print("Frozen grouped rank parameters:",full_rank.PARAMS,"rounds:",full_rank.ROUNDS)
summary=full_rank.run(DATA_ROOT,NEURAL_ROOT,output,source_path,weighting_path,Path("/kaggle/working/ctr-tabular-choice-003-setwise"))
summary

In [ ]:
from IPython.display import FileLink, display
from pathlib import Path
import json, zipfile
print("Experiment:", RUN_NAME)
report = json.loads((output / "summary.json").read_text())
for result in report["reports"]:
    print("Day", result["day"], "Adjusted AUC:", round(result["history_adjusted_auc"], 9), "Ordinary AUC:", round(result["ordinary_auc"], 9), "Gate:", result["positive_gate"])
print("Target 0.83 reached:", report["target_achieved"])
archive = Path("/kaggle/working") / (RUN_NAME + "-results.zip")
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as z:
    for path in output.rglob("*"):
        if path.is_file():
            z.write(path, path.relative_to(output))
display(FileLink(str(archive)))
